In [ ]:
"""
OJ 287 — R.O.M. honest fit
- β_spin dropped (Anton's instruction)
- ω_shift(o) = α·o with α = τ_Y²/(1-e²);  τ_Y² = 3β² - 2β⁴ (closure κ²=2β²)
- T in the fit = anomalistic (radial) period
- 5 free parameters: β_orb, e, ω_i, T_rad_days, t_0
- No σ assumed upfront; residuals reported in days; χ² bracketed
- Keplerian (α=0) fit as baseline
"""
import numpy as np
from scipy.optimize import differential_evolution, minimize

C_KMS        = 299792.458
MSUN_GEOM_KM = 1.4766      # GM_sun/c²  in km

flares_yr = [1971.13, 1972.97, 1983.00, 1984.13, 1994.89,
             1995.84, 2005.82, 2007.70, 2015.87, 2019.57]
obs_mjd   = np.array([(y-1970.0)*365.25 + 40587.0 for y in flares_yr])

def predict_flares(params, with_precession=True):
    beta, e, omega_i, T_rad, t_0 = params
    if not (1e-4 < beta < 0.5 and 0.01 < e < 0.98):
        return None
    if with_precession:
        tau_Y_sq = 3*beta**2 - 2*beta**4
        alpha    = tau_Y_sq/(1 - e**2)
        if alpha >= 0.95:
            return None
    else:
        alpha = 0.0
    # Disk crossings at inertial angle o = k·π - ω_i (two per revolution)
    k = np.arange(-50, 50)
    o_k = k*np.pi - omega_i
    O_k = o_k*(1 - alpha)              # precessing-frame anomaly
    O_mod   = O_k % (2*np.pi)
    n_orbit = np.floor(O_k/(2*np.pi))
    # Kepler equation on O:  O (true anomaly) → E → M
    E_mod = 2*np.arctan(np.sqrt((1-e)/(1+e))*np.tan(O_mod/2))
    E_mod = np.where(E_mod < 0, E_mod + 2*np.pi, E_mod)
    M_mod = E_mod - e*np.sin(E_mod)
    M_tot = n_orbit*2*np.pi + M_mod
    t_pred = t_0 + (M_tot/(2*np.pi))*T_rad
    return np.sort(t_pred)

def ssd(params, with_precession=True):
    t_pred = predict_flares(params, with_precession)
    if t_pred is None:
        return 1e12
    return float(np.sum([np.min((t_pred - o)**2) for o in obs_mjd]))

def residuals_days(params, with_precession=True):
    t_pred = predict_flares(params, with_precession)
    if t_pred is None:
        return np.full(len(obs_mjd), np.inf)
    return np.array([t_pred[np.argmin(np.abs(t_pred-o))] - o for o in obs_mjd])

def run_fit(with_precession, label):
    print("="*76)
    print(f"{label}")
    print("="*76)
    bounds = [(0.05, 0.25), (0.3, 0.90), (0, 2*np.pi), (3800, 5000), (40000, 50000)]
    best_ssd = np.inf; best_x = None
    # Multi-start to avoid local minima
    for seed in [42, 7, 101, 2024, 13]:
        res_de = differential_evolution(
            ssd, bounds, args=(with_precession,), seed=seed, tol=1e-12,
            maxiter=6000, popsize=50, workers=1, polish=False)
        res = minimize(ssd, res_de.x, args=(with_precession,),
                       method='Nelder-Mead',
                       options={'xatol':1e-10, 'fatol':1e-12, 'maxiter':200000})
        if res.fun < best_ssd:
            best_ssd = res.fun; best_x = res.x
    params = best_x
    beta, e, omega_i, T_rad, t_0 = params
    r = residuals_days(params, with_precession)
    rms  = np.sqrt(np.mean(r**2))
    absmax = np.max(np.abs(r))
    print(f"β_orb = {beta:.6f}")
    print(f"e     = {e:.6f}")
    print(f"ω_i   = {omega_i:.4f} rad  ({np.degrees(omega_i):.2f}°)")
    print(f"T_rad = {T_rad:.2f} days  = {T_rad/365.25:.4f} yr  (anomalistic)")
    print(f"t_0   = {t_0:.2f} MJD")
    if with_precession:
        tau_Y_sq = 3*beta**2 - 2*beta**4
        alpha    = tau_Y_sq/(1-e**2)
        Dphi_per_radial = 2*np.pi*alpha/(1-alpha)       # advance of periapsis per radial cycle
        Dphi_per_inertial = 2*np.pi*alpha                # per inertial revolution
        print(f"α     = {alpha:.6f}")
        print(f"Δφ per radial cycle    = {np.degrees(Dphi_per_radial):.3f}°")
        print(f"Δφ per inertial cycle  = {np.degrees(Dphi_per_inertial):.3f}°")
    T_sec = T_rad*86400
    R_s_km  = T_sec*C_KMS*beta**3/np.pi
    M_total = (R_s_km/2)/MSUN_GEOM_KM
    print(f"R_s = T·c·β³/π = {R_s_km:.4e} km")
    print(f"M_total = R_s/(2·GM_sun/c²) = {M_total/1e10:.4f} × 10¹⁰ M☉")
    print()
    print(f"{'year_obs':>10}  {'year_pred':>10}  {'residual[d]':>12}")
    for y_obs, resid in zip(flares_yr, r):
        y_pred = y_obs + resid/365.25
        print(f"{y_obs:10.2f}  {y_pred:10.2f}  {resid:+12.2f}")
    print(f"\nRMS  residual: {rms:.2f} days")
    print(f"Max |residual|: {absmax:.2f} days")
    print(f"Σ r²  (SSD)  : {best_ssd:.2f} days²")
    return params, r, rms, best_ssd

print()
p_rom, r_rom, rms_rom, ssd_rom = run_fit(True,  "FIT A — R.O.M. with precession (5 params)")
print()
p_kep, r_kep, rms_kep, ssd_kep = run_fit(False, "FIT B — Keplerian baseline, α=0 (5 params)")

# Honest χ² comparison under several σ
print()
print("="*76)
print("χ² comparison under transparent σ assumptions")
print("="*76)
print(f"{'σ(days)':>8} {'χ²_ROM':>10} {'χ²_Kep':>10} {'Δχ²':>10}  "
      f"{'ROM<Kep?':>10}")
for sigma in [1.0, 2.0, 3.0, 5.0, 10.0]:
    cr = ssd_rom/sigma**2; ck = ssd_kep/sigma**2
    better = "yes" if cr < ck else "no"
    print(f"{sigma:8.1f} {cr:10.2f} {ck:10.2f} {ck-cr:+10.2f}  {better:>10}")
print()
print(f"SSD ratio (Kep/ROM) = {ssd_kep/ssd_rom:.3f}")
print(f"RMS ratio (Kep/ROM) = {rms_kep/rms_rom:.3f}")
print()
print("Interpretation note:")
print("  ΣΔt² depends on model quality, but χ² numerical value depends on σ.")
print("  Ratio is σ-independent. It tells whether precession genuinely helps.")
print("  If ratio ≈ 1, precession adds no predictive value over pure Kepler.")
print("  If ratio >> 1, R.O.M. precession is genuinely a better predictor.")


FIT A — R.O.M. with precession (5 params)
β_orb = 0.243431
e     = 0.895672
ω_i   = 1.3903 rad  (79.66°)
T_rad = 4211.86 days  = 11.5314 yr  (anomalistic)
t_0   = 49827.86 MJD
α     = 0.863386
Δφ per radial cycle    = 2275.159°
Δφ per inertial cycle  = 310.819°
R_s = T·c·β³/π = 5.0094e+11 km
M_total = R_s/(2·GM_sun/c²) = 16.9626 × 10¹⁰ M☉

  year_obs   year_pred   residual[d]
   1971.13     1971.14         +1.92
   1972.97     1972.96         -5.38
   1983.00     1983.08        +29.25
   1984.13     1984.13         -0.99
   1994.89     1994.84        -17.55
   1995.84     1995.82         -7.87
   2005.82     2005.85         +9.20
   2007.70     2007.63        -27.03
   2015.87     2015.83        -14.40
   2019.57     2019.66        +32.85

RMS  residual: 18.33 days
Max |residual|: 32.85 days
Σ r²  (SSD)  : 3361.19 days²

FIT B — Keplerian baseline, α=0 (5 params)
β_orb = 0.217557
e     = 0.591721
ω_i   = 1.1394 rad  (65.28°)
T_rad = 4183.39 days  = 11.4535 yr  (anomalistic)
t_0   = 45

In [ ]:
"""
OJ 287 — R.O.M. honest fit
- β_spin dropped (Anton's instruction)
- ω_shift(o) = α·o with α = τ_Y²/(1-e²);  τ_Y² = 3β² - 2β⁴ (closure κ²=2β²)
- T in the fit = anomalistic (radial) period
- 5 free parameters: β_orb, e, ω_i, T_rad_days, t_0
- No σ assumed upfront; residuals reported in days; χ² bracketed
- Keplerian (α=0) fit as baseline
"""
import numpy as np
from scipy.optimize import differential_evolution, minimize

C_KMS        = 299792.458
MSUN_GEOM_KM = 1.4766      # GM_sun/c²  in km

flares_yr = [1971.13, 1972.97, 1983.00, 1984.13, 1994.89,
             1995.84, 2005.82, 2007.70, 2015.87, 2019.57]
obs_mjd   = np.array([(y-1970.0)*365.25 + 40587.0 for y in flares_yr])

def predict_flares(params, with_precession=True):
    beta, e, omega_i, T_rad, t_0 = params
    if not (1e-4 < beta < 0.5 and 0.01 < e < 0.98):
        return None
    if with_precession:
        tau_Y_sq = 3*beta**2 - 2*beta**4
        alpha    = tau_Y_sq/(1 - e**2)
        if alpha >= 0.95:
            return None
    else:
        alpha = 0.0
    # Disk crossings at inertial angle o = k·π - ω_i (two per revolution)
    k = np.arange(-50, 50)
    o_k = k*np.pi - omega_i
    O_k = o_k*(1 - alpha)              # precessing-frame anomaly
    O_mod   = O_k % (2*np.pi)
    n_orbit = np.floor(O_k/(2*np.pi))
    # Kepler equation on O:  O (true anomaly) → E → M
    E_mod = 2*np.arctan(np.sqrt((1-e)/(1+e))*np.tan(O_mod/2))
    E_mod = np.where(E_mod < 0, E_mod + 2*np.pi, E_mod)
    M_mod = E_mod - e*np.sin(E_mod)
    M_tot = n_orbit*2*np.pi + M_mod
    t_pred = t_0 + (M_tot/(2*np.pi))*T_rad
    return np.sort(t_pred)

def ssd(params, with_precession=True):
    t_pred = predict_flares(params, with_precession)
    if t_pred is None:
        return 1e12
    return float(np.sum([np.min((t_pred - o)**2) for o in obs_mjd]))

def residuals_days(params, with_precession=True):
    t_pred = predict_flares(params, with_precession)
    if t_pred is None:
        return np.full(len(obs_mjd), np.inf)
    return np.array([t_pred[np.argmin(np.abs(t_pred-o))] - o for o in obs_mjd])

def run_fit(with_precession, label):
    print("="*76)
    print(f"{label}")
    print("="*76)
    bounds = [(0.05, 0.25), (0.3, 0.90), (0, 2*np.pi), (3800, 5000), (40000, 50000)]
    best_ssd = np.inf; best_x = None
    # Multi-start to avoid local minima
    for seed in [42, 7, 101]:
        res_de = differential_evolution(
            ssd, bounds, args=(with_precession,), seed=seed, tol=1e-12,
            maxiter=2000, popsize=30, workers=1, polish=False)
        res = minimize(ssd, res_de.x, args=(with_precession,),
                       method='Nelder-Mead',
                       options={'xatol':1e-10, 'fatol':1e-12, 'maxiter':50000})
        if res.fun < best_ssd:
            best_ssd = res.fun; best_x = res.x
    params = best_x
    beta, e, omega_i, T_rad, t_0 = params
    r = residuals_days(params, with_precession)
    rms  = np.sqrt(np.mean(r**2))
    absmax = np.max(np.abs(r))
    print(f"β_orb = {beta:.6f}")
    print(f"e     = {e:.6f}")
    print(f"ω_i   = {omega_i:.4f} rad  ({np.degrees(omega_i):.2f}°)")
    print(f"T_rad = {T_rad:.2f} days  = {T_rad/365.25:.4f} yr  (anomalistic)")
    print(f"t_0   = {t_0:.2f} MJD")
    if with_precession:
        tau_Y_sq = 3*beta**2 - 2*beta**4
        alpha    = tau_Y_sq/(1-e**2)
        Dphi_per_radial = 2*np.pi*alpha/(1-alpha)       # advance of periapsis per radial cycle
        Dphi_per_inertial = 2*np.pi*alpha                # per inertial revolution
        print(f"α     = {alpha:.6f}")
        print(f"Δφ per radial cycle    = {np.degrees(Dphi_per_radial):.3f}°")
        print(f"Δφ per inertial cycle  = {np.degrees(Dphi_per_inertial):.3f}°")
    T_sec = T_rad*86400
    R_s_km  = T_sec*C_KMS*beta**3/np.pi
    M_total = (R_s_km/2)/MSUN_GEOM_KM
    print(f"R_s = T·c·β³/π = {R_s_km:.4e} km")
    print(f"M_total = R_s/(2·GM_sun/c²) = {M_total/1e10:.4f} × 10¹⁰ M☉")
    print()
    print(f"{'year_obs':>10}  {'year_pred':>10}  {'residual[d]':>12}")
    for y_obs, resid in zip(flares_yr, r):
        y_pred = y_obs + resid/365.25
        print(f"{y_obs:10.2f}  {y_pred:10.2f}  {resid:+12.2f}")
    print(f"\nRMS  residual: {rms:.2f} days")
    print(f"Max |residual|: {absmax:.2f} days")
    print(f"Σ r²  (SSD)  : {best_ssd:.2f} days²")
    return params, r, rms, best_ssd

print()
p_rom, r_rom, rms_rom, ssd_rom = run_fit(True,  "FIT A — R.O.M. with precession (5 params)")
print()
p_kep, r_kep, rms_kep, ssd_kep = run_fit(False, "FIT B — Keplerian baseline, α=0 (5 params)")

# Honest χ² comparison under several σ
print()
print("="*76)
print("χ² comparison under transparent σ assumptions")
print("="*76)
print(f"{'σ(days)':>8} {'χ²_ROM':>10} {'χ²_Kep':>10} {'Δχ²':>10}  "
      f"{'ROM<Kep?':>10}")
for sigma in [1.0, 2.0, 3.0, 5.0, 10.0]:
    cr = ssd_rom/sigma**2; ck = ssd_kep/sigma**2
    better = "yes" if cr < ck else "no"
    print(f"{sigma:8.1f} {cr:10.2f} {ck:10.2f} {ck-cr:+10.2f}  {better:>10}")
print()
print(f"SSD ratio (Kep/ROM) = {ssd_kep/ssd_rom:.3f}")
print(f"RMS ratio (Kep/ROM) = {rms_kep/rms_rom:.3f}")
print()
print("Interpretation note:")
print("  ΣΔt² depends on model quality, but χ² numerical value depends on σ.")
print("  Ratio is σ-independent. It tells whether precession genuinely helps.")
print("  If ratio ≈ 1, precession adds no predictive value over pure Kepler.")
print("  If ratio >> 1, R.O.M. precession is genuinely a better predictor.")


FIT A — R.O.M. with precession (5 params)
β_orb = 0.251240
e     = 0.880568
ω_i   = 4.2091 rad  (241.16°)
T_rad = 4483.76 days  = 12.2759 yr  (anomalistic)
t_0   = 40841.68 MJD
α     = 0.807638
Δφ per radial cycle    = 1511.470°
Δφ per inertial cycle  = 290.750°
R_s = T·c·β³/π = 5.8626e+11 km
M_total = R_s/(2·GM_sun/c²) = 19.8518 × 10¹⁰ M☉

  year_obs   year_pred   residual[d]
   1971.13     1971.14         +2.34
   1972.97     1973.01        +13.23
   1983.00     1982.98         -6.38
   1984.13     1984.06        -24.59
   1994.89     1995.01        +42.43
   1995.84     1995.83         -4.06
   2005.82     2005.73        -31.33
   2007.70     2007.66        -15.01
   2015.87     2015.90        +11.89
   2019.57     2019.60        +11.48

RMS  residual: 20.31 days
Max |residual|: 42.43 days
Σ r²  (SSD)  : 4123.29 days²

FIT B — Keplerian baseline, α=0 (5 params)
β_orb = 0.093116
e     = 0.559994
ω_i   = 4.5477 rad  (260.57°)
T_rad = 4183.39 days  = 11.4535 yr  (anomalistic)
t_0   = 